# Barcelona Noise Prediction: Land Use Feature Engineering
This notebook processes the official Barcelona land use shapefile and calculates the percentage of commercial, residential, industrial, green, and recreational areas within 50m and 100m buffers of our street network.

In [6]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import pandas as pd

print("Loading spatial data...")
# Load data (ensure your file paths match your VS Code workspace)
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")
print(noise_streets.crs)  #CRS = Coordinate Reference System
print(noise_streets.shape)
print(noise_streets.columns.tolist())
print("Number of street segments:", len(noise_streets))

landuse = gpd.read_file("../layers/bcn_usossl_etrs89_shp.gpkg")
print(landuse.crs)  #CRS = Coordinate Reference System
print(landuse.shape)
print(landuse.columns.tolist())
print("Number of land use polygons:", len(landuse))

# Quick sanity check to see the loaded data
display(landuse.head(2))

Loading spatial data...
EPSG:25831
(15115, 30)
['TRAM', 'TOTAL_D', 'TOTAL_E', 'TOTAL_N', 'TOTAL_DEN', 'TRANSIT_D', 'TRANSIT_E', 'TRANSIT_N', 'TRANSIT_DEN', 'GI_TR_D', 'GI_TR_E', 'GI_TR_N', 'GI_TR_DEN', 'FFCC_D', 'FFCC_E', 'FFCC_N', 'FFCC_DEN', 'INDUST_D', 'INDUST_E', 'INDUST_N', 'INDUST_DEN', 'VIANANTS_D', 'VIANANTS_E', 'OCI_N', 'PATIS_D', 'PATIS_E', 'geometry_type', 'start', 'end', 'geometry']
Number of street segments: 15115
EPSG:25831
(14320, 14)
['FAMILIA', 'CLAU', 'District', 'NDistric', 'CBarri', 'NBarri', 'C_AEB', 'CSecCens', 'Area', 'Perimetre', 'Coord_X', 'Coord_Y', 'Descripcio', 'geometry']
Number of land use polygons: 14320


,FAMILIA,CLAU,District,NDistric,CBarri,NBarri,C_AEB,CSecCens,Area,Perimetre,Coord_X,Coord_Y,Descripcio,geometry
0,22@,13hs,10,Sant Martí,66,el Parc i la Llacuna del Poblenou,208,042,735.029735,128.008296,432836.027297,4.583821e+06,13hs - 22@,"MULTIPOLYGON (((432764.589 4583628.639, 432747..."
1,22@,22@,10,Sant Martí,66,el Parc i la Llacuna del Poblenou,208,043,5749.281523,536.159605,432603.509491,4.583503e+06,22@ - 22@,"MULTIPOLYGON (((432566.688 4583298.456, 432562..."


## Categorize Land Use and Align CRS
We map the Catalan zoning categories to our 5 machine-learning buckets. Then, we force both datasets into EPSG:25831 (ETRS89 / UTM zone 31N) so our buffer math is perfectly calculated in meters.

In [ ]:
zoning_to_ml_buckets = {
    '22@': 'commercial',
    'Casc Antic': 'residential',
    "Conservació de l'Estructura Urbana i Edificatòria": 'residential',
    'Densificació Urbana': 'residential',
    'Edificació aïllada subzones plurifamiliars': 'residential',
    'Edificació aïllada subzones unifamiliars': 'residential',
    'Habitatge dotacional i de protecció': 'residential',
    'Ordenació Volumètrica Específica': 'residential',
    'Remodelació Física': 'residential',
    'Renovació Urbana: Rehabilitació': 'residential',
    'Industrial': 'industrial',
    'Sistema de serveis tècnics': 'industrial',
    'Sistema ferroviari': 'industrial',
    'Sistema relatiu al port': 'industrial',
    'Protecció de Sistemes Generals i Vials': 'industrial',
    'Parc Forestal': 'green',
    'Parcs i Jardins Urbans': 'green',
    'Verd privat': 'green',
    "Transformació de l'ús existent (PARC)": 'green',
    'Equipaments Comunitaris i Dotacions': 'recreational',
    "Transformació de l'ús existent (EQUIPAMENT)": 'recreational' 
}

# Apply mapping
landuse['ml_category'] = landuse['FAMILIA'].map(zoning_to_ml_buckets).fillna('other')



## Define the Spatial Math Function
This function takes the street geometries, draws the buffer, intersects it with the official land use polygons, and pivots the calculated areas into percentage columns.

In [13]:
def calculate_landuse_pct(streets_gdf, landuse_gdf, buffer_size):
    print(f"Processing {buffer_size}m buffers...")
    
    # Create buffers and calculate total buffer area
    buffered = streets_gdf.copy()
    buffered['geometry'] = buffered.geometry.buffer(buffer_size)
    buffered['buffer_area'] = buffered.geometry.area
    
    # Intersect with landuse
    intersection = gpd.overlay(buffered, landuse_gdf, how='intersection')
    intersection['intersected_area'] = intersection.geometry.area
    
    # Group by street and category to get total area per category
    # NOTE: Change 'street_id' if your unique ID column is named something else!
    grouped = intersection.groupby(['TRAM', 'ml_category'])['intersected_area'].sum().reset_index()
    
    # Merge back to get the total buffer area and calculate percentage
    grouped = grouped.merge(buffered[['TRAM', 'buffer_area']], on='TRAM')
    grouped['pct'] = (grouped['intersected_area'] / grouped['buffer_area']) * 100
    
    # Pivot to wide format (Machine Learning ready)
    features = grouped.pivot(index='TRAM', columns='ml_category', values='pct').fillna(0)
    
    # Rename columns to include the buffer size
    features.columns = [f"{col}_pct_{buffer_size}m" for col in features.columns]
    
    return features.reset_index()

## Execute Buffer Calculations
Running the function for both 50m and 100m. This might take a minute or two depending on the density of the polygons.

In [14]:
features_50m = calculate_landuse_pct(noise_streets, landuse, 50)
features_100m = calculate_landuse_pct(noise_streets, landuse, 100)

display(features_50m.head(3))

Processing 50m buffers...
Processing 100m buffers...


,TRAM,commercial_pct_50m,green_pct_50m,industrial_pct_50m,recreational_pct_50m,residential_pct_50m
0,LRD0001,0.0,5.023397,18.526316,7.315722,2.592209
1,LRD0002,0.0,5.095533,20.054553,0.000000,9.871660
2,LRD0003,0.0,15.840463,0.000000,21.698592,2.127330


## Merge
Attach our new percentage features back to the original tabular street data.

In [ ]:
# Create a baseline dataframe without the geometry
final_df = pd.DataFrame(noise_streets.drop(columns=['geometry']))

# Attach the 50m and 100m features
final_df = final_df.merge(features_50m, on='TRAM', how='left')
final_df = final_df.merge(features_100m, on='TRAM', how='left')

# Fill any remaining NaNs with 0 
final_df = final_df.fillna(0)

NameError: name 'streets' is not defined